# Mixture of Experts: More Parameters, Same FLOPs

The runnable code of [Mixture of Experts: More Parameters, Same FLOPs](https://residual-stream.lvtan71.workers.dev/posts/mixture-of-experts/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/mixture-of-experts.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 1 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "mixture-of-experts"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in []:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## An MoE tiny GPT

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/mixture-of-experts/#an-moe-tiny-gpt)

In [ ]:
import torch
import torch.nn.functional as F

# A mixture-of-experts MLP: E small SwiGLU experts and a router. Each token
# goes to its top-k experts, weighted by the router's softmax probabilities
# (renormalized over the chosen k). Balancing, one of:
#   "aux":  the Switch Transformer loss, E * sum_e f_e * P_e, where f_e is the
#           share of tokens routed to expert e and P_e its mean router
#           probability; minimized when both are uniform (Fedus et al., 2022);
#   "bias": DeepSeek-V3's auxiliary-loss-free method: a per-expert bias added
#           to the scores for *choosing* experts only, nudged after each step
#           toward under-used experts (Wang et al., 2024);
#   "none": nothing.
class MoE(torch.nn.Module):
    def __init__(self, D, F_, E, k, balance, bias_rate=1e-3, renorm_top1=False):
        super().__init__()
        self.E, self.k, self.balance, self.bias_rate = E, k, balance, bias_rate
        self.renorm_top1 = renorm_top1
        self.router = torch.nn.Linear(D, E, bias=False)
        self.gate = torch.nn.Parameter(torch.randn(E, D, F_) * 0.02)
        self.up = torch.nn.Parameter(torch.randn(E, D, F_) * 0.02)
        self.down = torch.nn.Parameter(torch.randn(E, F_, D) * 0.02)
        self.register_buffer("bias", torch.zeros(E))
        self.aux_loss, self.load = torch.tensor(0.0), torch.zeros(E)

    def forward(self, x):
        B, T, D = x.shape
        x = x.reshape(-1, D)
        probs = self.router(x).softmax(-1)                                # (tokens, E)
        choose_by = probs + self.bias if self.balance == "bias" else probs
        top = choose_by.topk(self.k, -1).indices                          # (tokens, k)
        weight = probs.gather(1, top)
        if self.k > 1 or self.renorm_top1:
            weight = weight / weight.sum(-1, keepdim=True)        # renormalize over the chosen k
        # With k = 1, renormalizing makes every weight exactly 1, so the
        # router would get no gradient from the language-model loss at all;
        # Switch Transformers uses the raw probability instead.
        out = torch.zeros_like(x)
        counts = torch.zeros(self.E, device=x.device)
        for e in range(self.E):                                           # run each expert on its tokens
            rows, slot = (top == e).nonzero(as_tuple=True)
            counts[e] = len(rows)
            if len(rows):
                h = x[rows]
                y = (F.silu(h @ self.gate[e]) * (h @ self.up[e])) @ self.down[e]
                out.index_add_(0, rows, y * weight[rows, slot, None])
        f = counts / counts.sum()
        self.load = f.detach()
        self.aux_loss = self.E * (f * probs.mean(0)).sum() if self.balance == "aux" else torch.tensor(0.0, device=x.device)
        if self.balance == "bias" and self.training:                      # raise the bias of under-loaded experts
            with torch.no_grad():
                self.bias += self.bias_rate * torch.sign(1 / self.E - f)
        return out.reshape(B, T, D)

## Counting real models

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/mixture-of-experts/#counting-real-models)

In [ ]:
# Mixture-of-experts arithmetic from published configurations.
# Each expert is a SwiGLU MLP: 3 * D * F weights (gate, up, down).
def expert(D, F):
    return 3 * D * F

# Mixtral 8x7B: 32 layers, D = 4,096, 8 experts of F = 14,336, top-2.
D, L, F, E, k = 4096, 32, 14336, 8, 2
print(f"Mixtral 8x7B: one expert {expert(D, F) / 1e6:.0f}M weights; all experts, all layers "
      f"{expert(D, F) * E * L / 1e9:.1f}B; used per token {expert(D, F) * k * L / 1e9:.1f}B")

# DeepSeek-V3: 61 layers, the first 3 dense (F = 18,432); the other 58 have
# 256 routed experts of F = 2,048, top-8, plus 1 shared expert every token uses.
D, L, dense, F_dense, E, F, k, shared = 7168, 61, 3, 18432, 256, 2048, 8, 1
routed_all = expert(D, F) * E * (L - dense)
used = (expert(D, F) * (k + shared)) * (L - dense) + expert(D, F_dense) * dense
print(f"DeepSeek-V3: one routed expert {expert(D, F) / 1e6:.0f}M weights; routed experts, all layers "
      f"{routed_all / 1e9:.0f}B of the reported 671B; MLP weights used per token {used / 1e9:.1f}B of the reported 37B active")

# Expert parallelism: experts live on different GPUs, so every token's
# hidden vector travels to its k experts and the results travel back.
# DeepSeek-V3 dispatches in FP8 (1 byte) and combines in BF16 (2 bytes).
per_layer = k * D * (1 + 2)
print(f"all-to-all per token: {per_layer / 1024:.0f} KiB per MoE layer, {per_layer * (L - dense) / 2**20:.1f} MiB "
      f"across the {L - dense} MoE layers (at most; tokens whose experts share a GPU send less)")

# Decoding: a batch of B tokens, each picking k of E experts. How many
# experts does a step touch, and how many tokens does each one get? An
# expert that processes t tokens does 2 * t FLOPs per weight it reads.
for B in (1, 8, 64, 512, 4096):
    touched = E * (1 - (1 - k / E) ** B)
    print(f"batch {B:>5,}: about {touched:5.1f} of {E} experts touched per layer, "
          f"{B * k / touched:6.1f} tokens per touched expert")

The post printed:

```text
Mixtral 8x7B: one expert 176M weights; all experts, all layers 45.1B; used per token 11.3B
DeepSeek-V3: one routed expert 44M weights; routed experts, all layers 654B of the reported 671B; MLP weights used per token 24.2B of the reported 37B active
all-to-all per token: 168 KiB per MoE layer, 9.5 MiB across the 58 MoE layers (at most; tokens whose experts share a GPU send less)
batch     1: about   8.0 of 256 experts touched per layer,    1.0 tokens per touched expert
batch     8: about  57.4 of 256 experts touched per layer,    1.1 tokens per touched expert
batch    64: about 222.4 of 256 experts touched per layer,    2.3 tokens per touched expert
batch   512: about 256.0 of 256 experts touched per layer,   16.0 tokens per touched expert
batch 4,096: about 256.0 of 256 experts touched per layer,  128.0 tokens per touched expert
```